# Pyspark intro

a short note on notebooks in databricks - can choose between
- SQL notebook
- Python notebook

the choice makes all cells default to that choice .g. SQL -> cells become SQL by default

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_athletes

In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

# Infer the schema

we note that age, height becomes strings, that is because it ha N/A values.

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True)
df_athletes

#Define explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'").sort("NOC", "Medal").display()

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

# Todo: pick out swedish medals in table tennis
spark.sql("SELECT * FROM df_athletes_schema WHERE NOC = 'SWE' AND Sport = 'Table Tennis' AND Medal != 'NA'").display()

## count medals and plot

In [0]:
df_swe_medals = spark.sql("""
        SELECT
        sport,
        COUNT(*) AS medal_count
        FROM df_athletes_schema
        WHERE noc = 'SWE'
        AND medal IN ('Gold', 'Silver', 'Bronze')
        GROUP BY sport
        ORDER BY medal_count DESC
        """)

df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind = "bar", y="sport", x="medal_count")
fig.update

##Ingesting data to unity catalog

In [0]:
%sql
SHOW SCHEMAS IN DATA;

CREATE TABLE IF NOT EXISTS data.olympic_games.sweden_medals AS
(
    SELECT
        name,
        age,
        height,
        weight,
        year,
        sport,
        medal
    FROM df_athletes_schema
    WHERE noc = 'WE' AND medal IN ('Gold', 'Silver', 'Bronze')
)